# 3. Decay domain

Part of the **[Fig. 4 chapter](fig4.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

- `f'{ENTEX_ROOT}/L1color.tsv'`  ·  _metadata: color_
- `f'domain/L1_{mode}.boundary.h5ad'`  ·  _domain/boundary_
- `f'{indir}clustering/merged/5kCG100k3C_summary.h5ad'`  ·  _joint summary obj_
- `f'{indir}merged_cool_raw/L1/{ct}.mcool::resolutions/25000'`  ·  _other_
- `f'decay_domain/{ct}_histdomain_{mode}.npy'`  ·  _domain/boundary_


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import os
import time
import cooler
from glob import glob
from concurrent.futures import ProcessPoolExecutor, as_completed

import qnorm
import anndata
from scipy.stats import zscore

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [ ]:
indir = f'{ENTEX_ROOT}/'


In [ ]:
L1meta = pd.read_csv(f'{ENTEX_ROOT}/L1color.tsv', sep='\t', header=0, index_col=0)
L1meta['L1_annot'] = L1meta['L1_annot'].str.replace(' ','-').str.replace('/','_')


In [ ]:
res = 25000
chrom_size_path = '/large_experiments/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
mode = 'raw'
adata = anndata.read_h5ad(f'domain/L1_{mode}.boundary.h5ad')
adata = adata[:, adata.var['chrom'].isin(chrom_sizes.index)]
adata

In [ ]:
bin_tmp = adata.var.copy()
bin_tmp.index = bin_tmp.index.astype(int)
chrom_offset = {c: bin_tmp[bin_tmp['chrom'] == c].index[0] for c in chrom_sizes.index}


In [ ]:
domain_size = []
for i,ct in enumerate(adata.obs.index):
    tmp = adata.X[i]
    tmp = np.diff(np.repeat(tmp.indices, tmp.data))[0::2]
    tmp = pd.DataFrame(tmp[:, None], columns=['size'])
    tmp['L1'] = ct
    domain_size.append(tmp)

domain_size = pd.concat(domain_size, axis=0).reindex()

In [ ]:
domain_size['size'] = domain_size['size'] * res

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2), dpi=300)
sns.boxplot(data=domain_size, x='L1', y='size', order=L1meta.index, 
               showfliers=False, 
               palette=L1meta['color'].to_dict(), ax=ax)
ax.set_xticklabels(L1meta['L1_annot'], rotation=90)

In [ ]:
ws = 10
domain = {}
for i,ct in enumerate(adata.obs.index):
    domaintmp = adata.X.indices[adata.X.indptr[i]:adata.X.indptr[i+1]]
    domainfilter = (bin_tmp.loc[domaintmp, 'start'] >= (ws * res)) & (bin_tmp.loc[domaintmp, 'start'] < (bin_tmp.loc[domaintmp, 'chrom'].map(chrom_sizes).astype(int) - ws * res))
    domain[ct] = domaintmp[domainfilter]
    

In [ ]:
meta = anndata.read_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad').obs
leg = meta.groupby('L1')['Short/Long'].median().sort_values().index[::-1]


In [ ]:
def contact_hist(ct):
    count = np.zeros((ws*2, ws*2))
    cool = cooler.Cooler(f'{indir}merged_cool_raw/L1/{ct}.mcool::resolutions/25000')
    for i,c in enumerate(chrom_sizes.index):
        data = cool.matrix(balance=False, sparse=True).fetch(c).tocsr()
        domaintmp = domain[ct][bin_tmp.loc[domain[ct], 'chrom']==c] - chrom_offset[c]
        for xx in domaintmp:
            count += data[(xx-ws):(xx+ws), (xx-ws):(xx+ws)].toarray()
    return count
    

In [ ]:
cpu = 32
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for ct in L1meta.index:
        future = executor.submit(
            contact_hist,
            ct=ct,
        )
        futures[future] = ct
    count = {}
    for future in as_completed(futures):
        ct = futures[future]
        count[ct] = future.result()
        print(f'{ct} finished')


In [ ]:
ref = np.sum([count[ct] for ct in leg], axis=0)
# ref = ref / ref.sum()
ref = ref / meta['CisLongContact'].sum()

In [ ]:
ticks = [-0.5, 9.5, 19.5]
ticklabels = ['-250k', 'Boundary', '+250k']

fig, axes = plt.subplots(7, 5, figsize=(10, 14), sharex='all', sharey='all')
for i,ct in enumerate(leg):
    ax = axes.flatten()[i]
    tmp = count[ct].copy()
    # tmp = tmp / tmp.sum()
    tmp = tmp / meta.loc[meta['L1']==ct, 'CisLongContact'].sum()
    tmp = np.log2(tmp / ref)
    ax.imshow(tmp, cmap='vlag', interpolation='none', vmin=-1, vmax=1)
    ax.set_xticks(ticks)
    ax.set_xticklabels(ticklabels, rotation=90)
    ax.set_title(L1meta.loc[ct, 'L1_annot'])
    ax.set_yticks(ticks)
    ax.set_yticklabels(ticklabels)

plt.tight_layout()

In [ ]:
chrom1 = 1
chrom2 = 5
pos1 = 2
pos2 = 6
n_bins = adata.shape[1]

from scipy.sparse import csr_matrix
def contact_hist(contact_path, ct):
    count = np.zeros((ws*2, ws*2))
    data = pd.read_csv(contact_path, sep='\t', header=None, index_col=None)
    data = data.loc[(data[chrom1]==data[chrom2]) & data[chrom1].isin(chrom_sizes.index) & (np.abs(data[pos1] - data[pos2]) > 2500)]
    data[pos1] = (data[pos1] - 1) // res + data[chrom1].map(chrom_offset).astype(int)
    data[pos2] = (data[pos2] - 1) // res + data[chrom1].map(chrom_offset).astype(int)
    data = data.loc[np.abs(data[pos2] - data[pos1]) <= (ws*2)]
    data = data.groupby(by=[pos1, pos2])[chrom1].count().reset_index()
    data = csr_matrix((data[chrom1].astype(np.int32), (data[pos1], data[pos2])), (n_bins, n_bins))
    for xx in domain[ct]:
        count += data[(xx-ws):(xx+ws), (xx-ws):(xx+ws)].toarray()
    return count
   

In [ ]:
cpu = 32
for ct in L1meta.index:
    count = np.zeros((ws*2, ws*2))
    with ProcessPoolExecutor(cpu) as executor:
        futures = {}
        for cell in meta.index[meta['L1']==ct]:
            future = executor.submit(
                contact_hist,
                contact_path=f'{indir}contact/{cell}.3C.contact.rmbkl.tsv.gz',
                ct=ct,
            )
            futures[future] = cell
        for future in as_completed(futures):
            cell = futures[future]
            xx = future.result()
            count += xx
            # print(f'{cell} finished')

    np.save(f'decay_domain/{ct}_histdomain_{mode}.npy', count)
    print(f'{ct} finished')


In [ ]:
mode = 'raw'
count = []
for ct in leg:
    tmpcount = np.load(f'decay_domain/{ct}_histdomain_{mode}.npy')
    tmpcount = tmpcount + tmpcount.T
    count.append(tmpcount)
    

In [ ]:
ref = np.sum(count, axis=0)
# ref = ref / ref.sum()
ref = ref / meta['CisLongContact'].sum()

In [ ]:
ticks = [-0.5, 9.5, 19.5]
ticklabels = ['-250k', 'Boundary', '+250k']

fig, axes = plt.subplots(7, 5, figsize=(10, 14), sharex='all', sharey='all')
for i,ct in enumerate(leg):
    ax = axes.flatten()[i]
    tmp = count[i].copy()
    # tmp = tmp / tmp.sum()
    tmp = tmp / meta.loc[meta['L1']==ct, 'CisLongContact'].sum()
    tmp = np.log2(tmp / ref)
    ax.imshow(tmp, cmap='vlag', interpolation='none', vmin=-1, vmax=1)
    ax.set_xticks(ticks)
    ax.set_xticklabels(ticklabels, rotation=90)
    ax.set_title(L1meta.loc[ct, 'L1_annot'])
    ax.set_yticks(ticks)
    ax.set_yticklabels(ticklabels)

plt.tight_layout()